# 🤖 Multi-Agent Job Search System
### CrewAI + LangChain + Groq (llama-3.3-70b-versatile)

**Agents:** Profile Analyst → Job Researcher → Resume Specialist → Cover Letter Writer → Interview Coach → Report Compiler

**Tools:** Real-time job search (Serper), Salary research, Company intelligence, Skill-gap analyzer (FAISS), Interview question bank


## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -q crewai crewai-tools langchain langchain-groq langchain-community \
    faiss-cpu sentence-transformers google-search-results \
    python-dotenv pydantic rich tabulate requests
print('✅ All packages installed!')


## 🔑 Step 2 — API Keys

- **Groq** (free): https://console.groq.com
- **Serper** (free 2 500 calls): https://serper.dev


In [ ]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY']   = getpass('Groq API key: ')
os.environ['SERPER_API_KEY'] = getpass('Serper API key: ')

assert len(os.environ['GROQ_API_KEY'])   > 10, 'Invalid Groq key'
assert len(os.environ['SERPER_API_KEY']) > 10, 'Invalid Serper key'
print('✅ Keys configured!')


## 🧠 Step 3 — Imports & LLM Setup

In [ ]:
import os, json, warnings, textwrap
from datetime import datetime
warnings.filterwarnings('ignore')

from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from pydantic import BaseModel, Field
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.markdown import Markdown

console = Console()

# Analytical LLM — low temperature for factual accuracy
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.1,
    max_tokens=4096,
    groq_api_key=os.environ['GROQ_API_KEY']
)

# Creative LLM — higher temperature for cover letters
creative_llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.6,
    max_tokens=4096,
    groq_api_key=os.environ['GROQ_API_KEY']
)

console.print('[bold cyan]LLM ready: llama-3.3-70b-versatile via Groq[/bold cyan]')


## 🔧 Step 4 — Custom Tools

In [ ]:
# ── Tool 1: Job Search ──────────────────────────────────────────
class JobSearchTool(BaseTool):
    name: str = 'job_search'
    description: str = (
        'Search for real job listings online. '
        'Input: job title + location, e.g. "Python ML Engineer remote". '
        'Returns current postings with requirements.'
    )
    def _run(self, query: str) -> str:
        try:
            search = GoogleSerperAPIWrapper(
                serper_api_key=os.environ['SERPER_API_KEY'], type='search', k=5)
            r = search.run(
                f'{query} job opening site:linkedin.com OR site:indeed.com OR site:glassdoor.com')
            return r or 'No results. Try a different query.'
        except Exception as e:
            try:
                return DuckDuckGoSearchRun().run(f'{query} job posting 2024 requirements')
            except:
                return f'Search error: {e}'


# ── Tool 2: Salary Research ──────────────────────────────────────
class SalaryResearchTool(BaseTool):
    name: str = 'salary_research'
    description: str = (
        'Research salary ranges for a role/location. '
        'Returns salary benchmarks and negotiation tips.'
    )
    def _run(self, query: str) -> str:
        try:
            search = GoogleSerperAPIWrapper(
                serper_api_key=os.environ['SERPER_API_KEY'], type='search', k=3)
            return search.run(f'{query} salary range 2024 glassdoor OR levels.fyi OR payscale')
        except:
            return DuckDuckGoSearchRun().run(f'{query} average salary 2024')


# ── Tool 3: Company Research ─────────────────────────────────────
class CompanyResearchTool(BaseTool):
    name: str = 'company_research'
    description: str = (
        'Research a company culture, mission, products, and recent news. '
        'Input: company name.'
    )
    def _run(self, company: str) -> str:
        try:
            search = GoogleSerperAPIWrapper(
                serper_api_key=os.environ['SERPER_API_KEY'], type='search', k=3)
            return search.run(f'{company} company culture mission values recent news 2024')
        except:
            return DuckDuckGoSearchRun().run(f'{company} company overview culture values')


# ── Tool 4: Skill Gap Analyzer (semantic similarity) ─────────────
class SkillGapAnalyzerTool(BaseTool):
    name: str = 'skill_gap_analyzer'
    description: str = (
        'Analyze gap between candidate skills and job requirements using semantic similarity. '
        'Input JSON: {"candidate_skills": [...], "job_requirements": [...]}. '
        'Returns match percentage, matched skills, and gaps.'
    )
    def _run(self, input_str: str) -> str:
        try:
            data = json.loads(input_str)
            cands = data.get('candidate_skills', [])
            reqs  = data.get('job_requirements', [])
            if not cands or not reqs:
                return 'Provide both candidate_skills and job_requirements lists.'
            from sentence_transformers import SentenceTransformer, util
            model = SentenceTransformer('all-MiniLM-L6-v2')
            ce = model.encode(cands, convert_to_tensor=True)
            re = model.encode(reqs,  convert_to_tensor=True)
            matched, missing = [], []
            threshold = 0.55
            for i, req in enumerate(reqs):
                scores = util.cos_sim(re[i], ce)[0]
                best   = float(scores.max())
                bm     = cands[int(scores.argmax())]
                if best >= threshold:
                    matched.append({'requirement': req, 'matched_skill': bm, 'score': round(best, 3)})
                else:
                    missing.append({'requirement': req, 'closest': bm, 'gap': round(1 - best, 3)})
            pct = round(len(matched) / len(reqs) * 100, 1)
            rec = ('Strong match! Apply immediately.' if pct >= 70
                   else 'Good match. Highlight transferable skills.' if pct >= 50
                   else 'Consider upskilling before applying.')
            return json.dumps({'overall_match_pct': pct, 'matched': matched,
                               'gaps': missing, 'recommendation': rec}, indent=2)
        except json.JSONDecodeError:
            return 'Error: Input must be valid JSON.'
        except Exception as e:
            return f'Skill gap error: {e}'


# ── Tool 5: Interview Questions ──────────────────────────────────
class InterviewQuestionsTool(BaseTool):
    name: str = 'interview_questions'
    description: str = (
        'Fetch common interview questions for a role. '
        'Input: role name. Returns technical, behavioral, situational questions.'
    )
    def _run(self, role: str) -> str:
        try:
            search = GoogleSerperAPIWrapper(
                serper_api_key=os.environ['SERPER_API_KEY'], type='search', k=3)
            return search.run(f'{role} interview questions 2024 technical behavioral')
        except:
            return DuckDuckGoSearchRun().run(f'{role} common interview questions')


job_search_tool    = JobSearchTool()
salary_tool        = SalaryResearchTool()
company_tool       = CompanyResearchTool()
skill_gap_tool     = SkillGapAnalyzerTool()
interview_tool     = InterviewQuestionsTool()

console.print('[bold green]5 tools initialized![/bold green]')


## 👥 Step 5 — Define 6 Specialized Agents

In [ ]:
profile_analyst = Agent(
    role='Senior Career Profile Analyst',
    goal=(
        'Deeply analyze the candidate background, extract key skills, '
        'identify unique value propositions, detect transferable skills, '
        'and define a precise career positioning strategy.'
    ),
    backstory=(
        'You are a veteran career strategist with 15 years at top executive search firms. '
        'You have helped 5000+ professionals land dream jobs. You excel at identifying '
        'hidden strengths, quantifying achievements, and positioning candidates to stand '
        'out in competitive markets.'
    ),
    tools=[skill_gap_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

job_researcher = Agent(
    role='Elite Job Market Intelligence Researcher',
    goal=(
        'Find the top 5 best-fit job opportunities from real job boards, '
        'analyze job descriptions, evaluate company fit, research salary ranges, '
        'and rank opportunities by match score and growth potential.'
    ),
    backstory=(
        'You are a data-driven job market analyst combining web intelligence with deep '
        'knowledge of hiring trends. You know how to find hidden opportunities, evaluate '
        'culture signals, and spot red flags in job descriptions.'
    ),
    tools=[job_search_tool, salary_tool, company_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=8
)

resume_specialist = Agent(
    role='ATS-Optimized Resume Tailoring Expert',
    goal=(
        'Craft highly tailored, ATS-optimized resumes. '
        'Strategically weave in keywords, quantify achievements, '
        'and structure the resume to pass automated screening systems.'
    ),
    backstory=(
        'You are a certified professional resume writer (CPRW) who has reverse-engineered '
        '200+ ATS systems. Your resumes consistently achieve 95%+ ATS pass rates.'
    ),
    tools=[skill_gap_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

cover_letter_writer = Agent(
    role='Persuasive Cover Letter Storytelling Expert',
    goal=(
        'Write compelling, personalized cover letters that tell a powerful career story '
        'and connect the candidate journey to the company mission.'
    ),
    backstory=(
        'You are a former journalist turned career coach. Your cover letters tell stories '
        'that make hiring managers lean forward. Your letters have a 78% interview '
        'conversion rate.'
    ),
    tools=[company_tool],
    llm=creative_llm,
    verbose=True,
    allow_delegation=False,
    max_iter=4
)

interview_coach = Agent(
    role='Executive Interview Preparation Coach',
    goal=(
        'Prepare candidates with role-specific interview questions, STAR-method '
        'frameworks, salary negotiation tactics, and red flag questions to ask.'
    ),
    backstory=(
        'You are a former FAANG engineering manager who has conducted 2000+ interviews '
        'and coached 800+ candidates to success. You know every trick interviewers use.'
    ),
    tools=[interview_tool, company_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

report_compiler = Agent(
    role='Strategic Career Action Plan Compiler',
    goal=(
        'Synthesize all research and coaching insights into a comprehensive, '
        'prioritized, and actionable career action plan with timelines and metrics.'
    ),
    backstory=(
        'You are a strategic consultant who specializes in turning complex career '
        'research into crystal-clear action plans. Your plans are comprehensive '
        'yet actionable, backed by data and clear success criteria.'
    ),
    tools=[],
    llm=llm,
    verbose=True,
    allow_delegation=True,
    max_iter=4
)

console.print('[bold green]6 agents created![/bold green]')


## 📋 Step 6 — Candidate Profile

> **Edit this cell with the real candidate information before running.**


In [ ]:
CANDIDATE_PROFILE = {
    'name': 'Alex Johnson',
    'location': 'San Francisco, CA (Open to Remote)',
    'target_role': 'Senior Machine Learning Engineer',
    'target_industry': 'AI/ML, FinTech, or HealthTech',
    'experience_years': 5,
    'current_role': 'ML Engineer at a Series B startup',
    'education': [
        'M.S. Computer Science (ML focus) - Stanford University, 2019',
        'B.S. Mathematics and Statistics - UC Berkeley, 2017'
    ],
    'technical_skills': [
        'Python', 'PyTorch', 'TensorFlow', 'Scikit-learn',
        'HuggingFace Transformers', 'LLM fine-tuning', 'RAG systems',
        'MLflow', 'Kubeflow', 'Apache Spark', 'SQL', 'NoSQL',
        'AWS SageMaker', 'GCP Vertex AI', 'Docker', 'Kubernetes',
        'FastAPI', 'Apache Airflow', 'Pandas', 'NumPy'
    ],
    'key_projects': [
        'Real-time fraud detection model (XGBoost + LSTM) - reduced false positives by 34%, saving $2.1M annually',
        'RAG-based customer support chatbot using LangChain + GPT-4 - handled 60% of tier-1 tickets',
        'MLOps pipeline migration to Kubeflow - cut deployment time from 2 weeks to 4 hours',
        'Published paper on contrastive learning for low-resource NLP at EMNLP 2022'
    ],
    'soft_skills': [
        'Technical leadership', 'Cross-functional collaboration',
        'Mentoring junior engineers', 'Stakeholder communication',
        'Agile/Scrum', 'Data storytelling'
    ],
    'certifications': [
        'AWS Certified Machine Learning - Specialty',
        'Google Professional ML Engineer',
        'Deep Learning Specialization (deeplearning.ai)'
    ],
    'salary_expectation': '$180,000 - $230,000 base + equity',
    'work_preference': 'Remote-first or hybrid (max 2 days/week in office)',
    'career_goal': 'ML Engineering Manager or Staff ML Engineer within 2 years',
    'unique_value_props': [
        'Full-stack ML: from research to production-grade systems',
        'Track record of shipping models that drive measurable business impact',
        'Strong ML + software engineering hybrid'
    ]
}

table = Table(title='Candidate Profile', show_header=True, header_style='bold cyan')
table.add_column('Field', style='bold yellow', width=25)
table.add_column('Details', style='white')
for k, v in CANDIDATE_PROFILE.items():
    if isinstance(v, list):
        display_v = ', '.join(v[:3]) + (f' (+{len(v)-3} more)' if len(v) > 3 else '')
    else:
        display_v = str(v)
    table.add_row(k.replace('_', ' ').title(), display_v)
console.print(table)


## 📝 Step 7 — Define Tasks

In [ ]:
profile_str = json.dumps(CANDIDATE_PROFILE, indent=2)

task_profile_analysis = Task(
    description=(
        f'Conduct a comprehensive analysis of this candidate profile:\n{profile_str}\n\n'
        'Your analysis MUST include:\n'
        '1. Executive Summary (3-4 sentences)\n'
        '2. Top 5 Strongest Skills with evidence\n'
        '3. Unique Value Proposition\n'
        '4. Transferable Skills\n'
        '5. Career Positioning Strategy\n'
        '6. Potential Weaknesses and how to address them\n'
        f'7. Skill Gap Assessment using skill_gap_analyzer for {CANDIDATE_PROFILE["target_role"]}\n'
        '8. Personal Brand Statement (2-3 sentences)'
    ),
    agent=profile_analyst,
    expected_output=(
        'Detailed structured career profile analysis with all 8 sections. '
        'Specific, actionable insights backed by evidence. Minimum 600 words.'
    )
)

task_job_research = Task(
    description=(
        f'Research the job market for {CANDIDATE_PROFILE["target_role"]} positions.\n'
        f'Candidate: {CANDIDATE_PROFILE["name"]} | Location: {CANDIDATE_PROFILE["location"]}\n'
        f'Target: {CANDIDATE_PROFILE["target_role"]} in {CANDIDATE_PROFILE["target_industry"]}\n'
        f'Salary: {CANDIDATE_PROFILE["salary_expectation"]}\n\n'
        'REQUIRED ACTIONS:\n'
        '1. Use job_search tool to find 5+ real listings matching the profile\n'
        '2. Use salary_research tool to benchmark compensation\n'
        '3. Use company_research tool for at least 2 target companies\n'
        'For each job: company, role, key requirements, match score (0-100%), salary, pros/cons.\n'
        'Also provide: market conditions and 30/60/90 day search timeline.'
    ),
    agent=job_researcher,
    expected_output=(
        'Job market report with top 5 ranked opportunities, salary benchmarks, '
        'company profiles, and prioritized application list with match scores.'
    )
)

task_resume_tailoring = Task(
    description=(
        f'Create a tailored ATS-optimized resume for {CANDIDATE_PROFILE["name"]}.\n'
        f'Profile:\n{profile_str}\n\n'
        'Include:\n'
        '1. Professional Summary (3-4 lines, keyword-rich)\n'
        '2. Technical Skills Section (organized by category)\n'
        '3. Experience Section with quantified achievements and action verbs\n'
        '4. Projects Section (top 3 with impact metrics)\n'
        '5. Education and Certifications\n'
        '6. ATS Optimization Notes (which keywords were added and why)\n'
        '7. Tailoring Notes for the top job opportunity\n'
        'Use skill_gap_analyzer to identify gaps vs top job requirements.'
    ),
    agent=resume_specialist,
    expected_output=(
        'Complete formatted ATS-optimized resume with all 7 sections. '
        'Include tailoring notes. Minimum 800 words.'
    )
)

task_cover_letter = Task(
    description=(
        f'Write 2 compelling cover letters for {CANDIDATE_PROFILE["name"]}.\n'
        f'Current role: {CANDIDATE_PROFILE["current_role"]}\n'
        f'Target: {CANDIDATE_PROFILE["target_role"]}\n'
        f'Career goal: {CANDIDATE_PROFILE["career_goal"]}\n\n'
        'Use company_research tool for each company.\n'
        'Each letter MUST:\n'
        '1. Open with a compelling hook (NOT "I am applying for...")\n'
        '2. Connect experience to the company mission and products\n'
        '3. Tell ONE powerful achievement story using STAR method\n'
        '4. Show specific knowledge of the company\n'
        '5. Express genuine enthusiasm\n'
        '6. Close with a confident call-to-action\n'
        '7. Be 350-450 words\n'
        'Tone: Professional but personable, confident but not arrogant.'
    ),
    agent=cover_letter_writer,
    expected_output=(
        'Two complete cover letters (350-450 words each) highly personalized '
        'using company research. Include strategy notes for each letter.'
    )
)

task_interview_prep = Task(
    description=(
        f'Create a comprehensive interview preparation guide for {CANDIDATE_PROFILE["name"]}.\n'
        f'Role: {CANDIDATE_PROFILE["target_role"]} | '
        f'Experience: {CANDIDATE_PROFILE["experience_years"]} years\n'
        f'Salary target: {CANDIDATE_PROFILE["salary_expectation"]}\n'
        f'Goal: {CANDIDATE_PROFILE["career_goal"]}\n\n'
        'Use interview_questions and company_research tools.\n'
        'Include:\n'
        '1. Top 10 Technical Questions with model answers\n'
        '2. Top 5 Behavioral Questions with STAR-method frameworks\n'
        '3. One ML System Design Question with solution approach\n'
        '4. 5 Smart Questions to Ask Interviewers\n'
        '5. Salary Negotiation Script\n'
        '6. Red Flags to Watch For\n'
        '7. 30-Day Interview Prep Roadmap\n'
        '8. Mindset and Performance Tips'
    ),
    agent=interview_coach,
    expected_output=(
        'Comprehensive interview prep guide with all 8 sections. '
        'Technical answers must be detailed. STAR answers use actual candidate experience. '
        'Salary script specific to target range. Minimum 1000 words.'
    )
)

task_final_report = Task(
    description=(
        f'Compile all outputs into a master Career Action Plan for {CANDIDATE_PROFILE["name"]}.\n\n'
        'Structure:\n'
        '# Career Action Plan\n'
        '## Executive Career Summary\n'
        '## Market Intelligence Summary\n'
        '## Top 5 Job Opportunities (ranked by fit score)\n'
        '## Skills Assessment and Gap Analysis\n'
        '## Application Materials Checklist\n'
        '## 30/60/90 Day Job Search Timeline\n'
        '## Interview Preparation Highlights\n'
        '## Salary Negotiation Strategy\n'
        '## Success Metrics and KPIs\n'
        '## Immediate Next Actions (This Week)\n\n'
        'Make this highly specific, data-driven, and immediately actionable.'
    ),
    agent=report_compiler,
    expected_output=(
        'Comprehensive Career Action Plan with all sections populated. '
        'Specific, actionable, references findings from all previous tasks. '
        'Minimum 1200 words.'
    )
)

console.print('[bold green]6 tasks defined![/bold green]')


## 🚀 Step 8 — Run the Crew

> Expected runtime: **3–8 minutes**


In [ ]:
console.print(Panel(
    f'[bold yellow]Launching Multi-Agent Job Search Crew[/bold yellow]\n'
    f'Candidate : {CANDIDATE_PROFILE["name"]}\n'
    f'Target    : {CANDIDATE_PROFILE["target_role"]}\n'
    f'Model     : llama-3.3-70b-versatile (Groq)\n'
    f'Process   : Sequential (6 agents, 6 tasks)',
    border_style='yellow'
))

job_search_crew = Crew(
    agents=[profile_analyst, job_researcher, resume_specialist,
            cover_letter_writer, interview_coach, report_compiler],
    tasks=[task_profile_analysis, task_job_research, task_resume_tailoring,
           task_cover_letter, task_interview_prep, task_final_report],
    process=Process.sequential,
    verbose=True,
    memory=False,
    max_rpm=30
)

start_time = datetime.now()
print(f'Started: {start_time.strftime("%H:%M:%S")}')

result = job_search_crew.kickoff()

elapsed = (datetime.now() - start_time).seconds
console.print(f'[bold green]Done in {elapsed // 60}m {elapsed % 60}s[/bold green]')


## 📊 Step 9 — Display Results

In [ ]:
console.print(Panel('[bold cyan]FINAL CAREER ACTION PLAN[/bold cyan]', border_style='cyan'))
console.print(Markdown(str(result)))


## 💾 Step 10 — Save & Download Results

In [ ]:
import os, shutil
from google.colab import files

ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
out = f'job_search_{ts}'
os.makedirs(out, exist_ok=True)

task_map = [
    ('01_profile_analysis',   task_profile_analysis),
    ('02_job_research',       task_job_research),
    ('03_resume_tailored',    task_resume_tailoring),
    ('04_cover_letters',      task_cover_letter),
    ('05_interview_prep',     task_interview_prep),
    ('06_final_action_plan',  task_final_report),
]

for fname, task in task_map:
    output = getattr(task, 'output', None)
    text   = str(output.raw) if output and hasattr(output, 'raw') else str(output or '')
    with open(f'{out}/{fname}.md', 'w', encoding='utf-8') as f:
        f.write(f'# {fname.upper()}\n\nCandidate: {CANDIDATE_PROFILE["name"]}\n\n---\n\n{text}')
    print(f'Saved: {out}/{fname}.md')

with open(f'{out}/MASTER_PLAN.md', 'w', encoding='utf-8') as f:
    f.write(f'# MASTER CAREER ACTION PLAN\nCandidate: {CANDIDATE_PROFILE["name"]}\n\n{str(result)}')

zip_name = f'career_plan_{CANDIDATE_PROFILE["name"].replace(" ", "_")}_{ts}'
shutil.make_archive(zip_name, 'zip', out)
files.download(f'{zip_name}.zip')
print(f'Downloaded: {zip_name}.zip')


## 📈 Step 11 — Skill Gap Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer, util

JOB_REQUIREMENTS = [
    'Python programming',
    'Deep learning (PyTorch/TensorFlow)',
    'MLOps and model deployment',
    'LLM and Transformer models',
    'Cloud platforms (AWS/GCP)',
    'SQL and data engineering',
    'Distributed computing (Spark)',
    'CI/CD for ML pipelines',
    'A/B testing and experimentation',
    'Technical leadership',
    'Stakeholder communication',
    'Research and paper reading',
]

stmodel  = SentenceTransformer('all-MiniLM-L6-v2')
all_skills = CANDIDATE_PROFILE['technical_skills'] + CANDIDATE_PROFILE['soft_skills']
ce = stmodel.encode(all_skills, convert_to_tensor=True)
re = stmodel.encode(JOB_REQUIREMENTS, convert_to_tensor=True)

scores = [float(util.cos_sim(re[i], ce)[0].max()) for i in range(len(JOB_REQUIREMENTS))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#0d1117')
for ax in [ax1, ax2]:
    ax.set_facecolor('#161b22')

# Bar chart
colors = ['#2ea043' if s >= 0.55 else '#f78166' for s in scores]
bars   = ax1.barh(range(len(JOB_REQUIREMENTS)), scores, color=colors, height=0.7)
ax1.axvline(x=0.55, color='#f0e68c', linestyle='--', linewidth=1.5, label='Threshold')
ax1.set_yticks(range(len(JOB_REQUIREMENTS)))
ax1.set_yticklabels(JOB_REQUIREMENTS, color='white', fontsize=9)
ax1.set_xlabel('Semantic Match Score', color='white')
ax1.set_title('Skill Match vs Job Requirements', color='white', pad=12)
ax1.tick_params(colors='white')
ax1.set_xlim(0, 1)
for sp in ax1.spines.values(): sp.set_color('#30363d')
ax1.legend(facecolor='#161b22', labelcolor='white', fontsize=9)
for bar, s in zip(bars, scores):
    ax1.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
             f'{s:.2f}', va='center', color='white', fontsize=8)

# Radar chart
cats = ['Python/ML\nLibs', 'MLOps/\nDeploy', 'Cloud\nPlatforms',
        'LLM/NLP', 'Data\nEngineering', 'Leadership']
N = len(cats)
cand_s = [0.95, 0.90, 0.85, 0.92, 0.80, 0.75]
avg_s  = [0.75, 0.70, 0.75, 0.65, 0.70, 0.65]
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]
cand_s += cand_s[:1]
avg_s  += avg_s[:1]
ax2.set_theta_offset(np.pi / 2)
ax2.set_theta_direction(-1)
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(cats, color='white', fontsize=9)
ax2.set_ylim(0, 1)
ax2.set_facecolor('#161b22')
ax2.spines['polar'].set_color('#30363d')
ax2.grid(color='#30363d', linewidth=0.5)
ax2.plot(angles, cand_s, 'o-', color='#2ea043', linewidth=2, label='Candidate')
ax2.fill(angles, cand_s, alpha=0.25, color='#2ea043')
ax2.plot(angles, avg_s,  'o-', color='#f78166', linewidth=2, linestyle='--', label='Average')
ax2.fill(angles, avg_s,  alpha=0.15, color='#f78166')
ax2.set_title('Skill Radar', color='white', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1),
           facecolor='#161b22', labelcolor='white', fontsize=9)

overall = sum(scores) / len(scores)
matched = sum(1 for s in scores if s >= 0.55)
fig.suptitle(f'Overall Match: {overall:.1%}  |  {matched}/{len(JOB_REQUIREMENTS)} Requirements Met',
             color='#f0e68c', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{out}/skill_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Match: {overall:.1%}  |  {matched}/{len(JOB_REQUIREMENTS)} met')


## 🔄 Step 12 — Quick Single-Role Search (Optional)

In [ ]:
def quick_search(target_role: str, company: str = None):
    desc = (
        f'For candidate {CANDIDATE_PROFILE["name"]} with skills: '
        f'{chr(44).join(CANDIDATE_PROFILE["technical_skills"][:10])}.\n'
        f'Role: {target_role}\n'
        + (f'Company: {company}\n' if company else '')
        + '1. Search job openings (job_search tool)\n'
          '2. Research company if given (company_research tool)\n'
          '3. Analyze skill match\n'
          '4. Write a 3-paragraph cover letter opening\n'
          '5. List top 5 interview prep tips'
    )
    quick_crew = Crew(
        agents=[job_researcher, cover_letter_writer],
        tasks=[Task(description=desc, agent=job_researcher,
                    expected_output='Job listings, match score, cover letter opening, tips.')],
        process=Process.sequential, verbose=True, max_rpm=30
    )
    return quick_crew.kickoff()

# Example usage (uncomment to run):
# result2 = quick_search('Staff ML Engineer', 'OpenAI')
# console.print(Markdown(str(result2)))
print('quick_search() ready. Usage: quick_search("Staff ML Engineer", "OpenAI")')
